(templates)=
# Salidas Estructuradas en LangChain


Cuando interactuamos con un LLM (modelo de lenguaje grande), la respuesta por defecto es **texto libre**: un párrafo, una frase, una lista en lenguaje natural. Esto es útil para conversaciones, pero resulta un problema cuando necesitamos que la aplicación **procese esa respuesta de forma programática**.

Imagina que le pides al modelo que extraiga el nombre, edad y ciudad de un texto. Si responde con:

> "El nombre es Juan, tiene 30 años y vive en Madrid."

…tu código tendrá que parsear esa cadena con expresiones regulares o lógica frágil. Aquí es donde entran las **salidas estructuradas**: forzar al modelo a devolver datos en un formato predecible (JSON, objetos Pydantic, etc.).

LangChain ofrece **dos enfoques principales** para lograr esto:

| Enfoque | Cuándo usarlo |
|---|---|
| **Output Parsers** | Mayor control manual, compatibilidad amplia |
| **`with_structured_output`** | Forma moderna, integrada y recomendada |

---

## Mapa completo de parsers disponibles
```{index} parsers
```

LangChain organiza sus parsers en dos paquetes principales:

| Paquete | Parsers disponibles |
|---|---|
| `langchain_core` | `StrOutputParser`, `JsonOutputParser`, `PydanticOutputParser`, `XMLOutputParser`, `CommaSeparatedListOutputParser`, `NumberedListOutputParser`, `MarkdownListOutputParser`, `SimpleJsonOutputParser` |
| `langchain` (community) | `BooleanOutputParser`, `DatetimeOutputParser`, `EnumOutputParser`, `StructuredOutputParser`, `YamlOutputParser`, `PandasDataFrameOutputParser`, `RegexParser`, `RegexDictParser`, `OutputFixingParser`, `RetryOutputParser`, `RetryWithErrorOutputParser`, `CombiningOutputParser` |

---

## Parte 1: Parsers de texto básico

###  `StrOutputParser` — El más simple de todos
```{index} strOutputParser
```

Devuelve la respuesta del modelo como una cadena de texto limpia. Es el parser de referencia cuando no necesitas estructura, solo texto.

**¿Cuándo usarlo?** Chatbots, generación de texto libre, resúmenes, traducciones.

```python
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un poeta en español."),
    ("human", "Escribe un haiku sobre {tema}.")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# StrOutputParser convierte el AIMessage en una cadena de texto simple
cadena = prompt | modelo | StrOutputParser()

resultado = cadena.invoke({"tema": "el amanecer"})

print(type(resultado))   # <class 'str'>
print(resultado)
# "Luz que rompe el velo oscuro,
#  el mundo despierta en calma,
#  nace un nuevo sol."
```

> **Nota:** Sin `StrOutputParser`, la cadena devuelve un objeto `AIMessage`. El parser extrae únicamente el texto del campo `.content`.

---

## Parte 2: Parsers de listas

###  `CommaSeparatedListOutputParser` — Lista separada por comas
```{index} CommaSeparatedListOutputParser
```

Instruye al modelo para responder con elementos separados por comas y los convierte en una lista Python.

```python
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = CommaSeparatedListOutputParser()

# Ver qué instrucciones genera para el prompt:
print(parser.get_format_instructions())
# → "Your response should be a list of comma separated values,
#    eg: `foo, bar, baz`"

prompt = ChatPromptTemplate.from_messages([
    ("system", "{format_instructions}"),
    ("human", "Dame 5 {categoria} famosos.")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)

cadena = prompt | modelo | parser

resultado = cadena.invoke({"categoria": "compositores clásicos"})

print(resultado)
# ['Bach', 'Mozart', 'Beethoven', 'Chopin', 'Vivaldi']
print(type(resultado))  # <class 'list'>
```

---

### `NumberedListOutputParser` — Lista numerada
```{index} NumberedListOutputParser
```

Parsea respuestas formateadas como listas numeradas (1. item, 2. item...) y las convierte en una lista Python.

```python
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import NumberedListOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = NumberedListOutputParser()

print(parser.get_format_instructions())
# → "Your response should be a numbered list with each item on a new line.
#    For example: `\n1. foo\n2. bar\n3. baz`"

prompt = ChatPromptTemplate.from_messages([
    ("system", "{format_instructions}"),
    ("human", "¿Cuáles son los pasos para hacer una tortilla española?")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)

cadena = prompt | modelo | parser

pasos = cadena.invoke({})

for i, paso in enumerate(pasos, 1):
    print(f"  Paso {i}: {paso}")

# Paso 1: Pelar y cortar las patatas en láminas finas.
# Paso 2: Pochar las patatas en aceite de oliva a fuego medio.
# Paso 3: Batir los huevos y mezclar con las patatas pochadas.
# ...
```

---

### `MarkdownListOutputParser` — Lista Markdown con asteriscos
```{index} MarkdownListOutputParser
```

Parsea listas en formato Markdown (`* item` o `- item`) y las convierte en lista Python.

```python
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import MarkdownListOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = MarkdownListOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", """Dame 5 nombres creativos para una startup de tecnología.
Usa formato de lista Markdown con asterisco (* nombre), un nombre por línea.
No uses comillas ni backlashes.""")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
cadena = prompt | modelo | parser

nombres = cadena.invoke({})

print(nombres)
# ['NexaFlow', 'ByteForge', 'QuantumLeap Labs', 'SynergyOS', 'DataPulse']
print(type(nombres))  # <class 'list'>
```

> **Diferencia clave:** `CommaSeparatedList` → elementos en una línea separados por comas. `NumberedList` → elementos numerados. `MarkdownList` → elementos con `*` o `-` al inicio.

---

## Parte 3: Parsers JSON y estructurados

###  `JsonOutputParser` — Diccionario Python
```{index} JsonOutputParser
```

Devuelve la respuesta del modelo como un diccionario Python. Versión ligera sin validación de tipos.

```python
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = JsonOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde siempre en JSON válido. {format_instructions}"),
    ("human", "{pregunta}")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)

cadena = prompt | modelo | parser

resultado = cadena.invoke({
    "pregunta": "Dame los 3 planetas más grandes del sistema solar con su diámetro en km."
})

print(type(resultado))  # <class 'dict'>
print(resultado)
# {
#   "planetas": [
#     {"nombre": "Júpiter", "diametro_km": 139820},
#     {"nombre": "Saturno", "diametro_km": 116460},
#     {"nombre": "Urano",   "diametro_km": 50724}
#   ]
# }
```

---

### `SimpleJsonOutputParser` — Alias simplificado
```{index} SimpleJsonOutputParser
```

Es funcionalmente equivalente a `JsonOutputParser` pero con un nombre más descriptivo. Internamente ambos heredan de la misma clase base.

```python
from langchain_core.output_parsers.json import SimpleJsonOutputParser

parser = SimpleJsonOutputParser()
# Uso idéntico a JsonOutputParser
```

---

### `PydanticOutputParser` — El más potente con validación
```{index} PydanticOutputParser
```

Define la estructura con Pydantic. Ofrece validación de tipos, campos obligatorios/opcionales y descripciones que guían al modelo.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Optional

class Persona(BaseModel):
    nombre: str = Field(description="Nombre completo de la persona")
    edad: int = Field(description="Edad en años")
    ciudad: str = Field(description="Ciudad donde vive")
    profesion: Optional[str] = Field(
        default=None,
        description="Profesión o trabajo de la persona"
    )

parser = PydanticOutputParser(pydantic_object=Persona)

# Las instrucciones de formato incluyen el JSON Schema completo del modelo
print(parser.get_format_instructions()[:200])
# → "The output should be formatted as a JSON instance that conforms
#    to the JSON schema below..."

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extrae información de textos.\n{format_instructions}"),
    ("human", "Extrae la información:\n\n{texto}")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

resultado = cadena.invoke({
    "texto": "María García tiene 28 años, es ingeniera de software y vive en Barcelona."
})

print(type(resultado))      # <class '__main__.Persona'>
print(resultado.nombre)     # María García
print(resultado.edad)       # 28
print(resultado.model_dump())
# {'nombre': 'María García', 'edad': 28, 'ciudad': 'Barcelona', 'profesion': 'ingeniera de software'}
```

---

### `StructuredOutputParser` — Esquema con `ResponseSchema`
```{index} StructuredOutputParser
```

Alternativa a Pydantic que usa objetos `ResponseSchema` para definir el esquema. Más sencillo pero sin validación de tipos. Útil para modelos que no soportan JSON Schema completo.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import ChatPromptTemplate

# Definimos el esquema con ResponseSchema (nombre + descripción de cada campo)
response_schemas = [
    ResponseSchema(
        name="respuesta",
        description="Respuesta a la pregunta del usuario"
    ),
    ResponseSchema(
        name="fuente",
        description="Tipo de fuente donde encontrarías esta información (libro, web, enciclopedia...)"
    ),
    ResponseSchema(
        name="confianza",
        description="Nivel de confianza en la respuesta: alta, media o baja"
    )
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)

# Veamos las instrucciones generadas:
print(parser.get_format_instructions())
# → Genera instrucciones con un bloque ```json con el esquema esperado

prompt = ChatPromptTemplate.from_messages([
    ("system", "Responde preguntas con detalle.\n{format_instructions}"),
    ("human", "{pregunta}")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

resultado = cadena.invoke({
    "pregunta": "¿Cuántos huesos tiene el cuerpo humano adulto?"
})

print(type(resultado))  # <class 'dict'>
print(resultado)
# {
#   'respuesta': 'El cuerpo humano adulto tiene 206 huesos.',
#   'fuente': 'Libro de anatomía o enciclopedia médica',
#   'confianza': 'alta'
# }
```

> **Diferencia con Pydantic:** `StructuredOutputParser` no valida tipos de datos. Si esperas un número pero el modelo devuelve texto, no lanzará error. Pydantic sí lo haría con un mensaje claro.

---

### `XMLOutputParser` — Formato XML
```{index} XMLOutputParser
```

Parsea respuestas en formato XML y las convierte en un diccionario Python con la estructura del árbol XML.

```python
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = XMLOutputParser(tags=["libro", "titulo", "autor", "año", "genero"])

prompt = ChatPromptTemplate.from_messages([
    ("system", """Responde en formato XML válido con esta estructura exacta:
<libro>
  <titulo>...</titulo>
  <autor>...</autor>
  <año>...</año>
  <genero>...</genero>
</libro>"""),
    ("human", "Dame información sobre el libro: {libro}")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

resultado = cadena.invoke({"libro": "Don Quijote de la Mancha"})

print(type(resultado))   # <class 'dict'>
print(resultado)
# {
#   'libro': [
#     {'titulo': ['Don Quijote de la Mancha']},
#     {'autor': ['Miguel de Cervantes']},
#     {'año': ['1605']},
#     {'genero': ['Novela de caballerías / Sátira']}
#   ]
# }
print(resultado['libro'][0]['titulo'][0])  # Don Quijote de la Mancha
```

> **¿Cuándo usar XML en lugar de JSON?** Cuando trabajas con sistemas legacy que consumen XML, o con el modelo Claude de Anthropic que tiende a producir XML de forma natural.

---

### `YamlOutputParser` — Formato YAML
```{index} YamlOutputParser
```

Similar a `PydanticOutputParser` pero el modelo responde en YAML en lugar de JSON. Requiere un modelo Pydantic como esquema.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import YamlOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List

class ConfiguracionServidor(BaseModel):
    nombre: str = Field(description="Nombre del servidor")
    host: str = Field(description="Dirección IP o hostname")
    puerto: int = Field(description="Número de puerto")
    protocolo: str = Field(description="Protocolo: http o https")
    servicios: List[str] = Field(description="Lista de servicios activos")

parser = YamlOutputParser(pydantic_object=ConfiguracionServidor)

print(parser.get_format_instructions()[:150])
# → "Return a YAML object that matches the following JSON schema..."

prompt = ChatPromptTemplate.from_messages([
    ("system", "Genera configuraciones de servidor.\n{format_instructions}"),
    ("human", "Crea una configuración para un servidor web de producción llamado '{nombre}'")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

config = cadena.invoke({"nombre": "web-prod-01"})

print(type(config))           # <class '__main__.ConfiguracionServidor'>
print(config.nombre)          # web-prod-01
print(config.puerto)          # 443
print(config.protocolo)       # https
print(config.servicios)       # ['nginx', 'ssl', 'load-balancer']
```

---

## Parte 4: Parsers de tipos primitivos

### `BooleanOutputParser` — Sí / No → `True` / `False`
```{index} BooleanOutputParser
```

Parsea respuestas binarias del modelo (YES/NO, SÍ/NO) y las convierte en un booleano Python. Muy útil para clasificaciones binarias, validaciones o decisiones.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import BooleanOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = BooleanOutputParser()

# Por defecto espera "YES" o "NO"
print(parser.true_val)   # YES
print(parser.false_val)  # NO

prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un clasificador de contenido.
Responde ÚNICAMENTE con YES si el texto es spam, o NO si no lo es.
No añadas ninguna explicación."""),
    ("human", "Clasifica este mensaje: {mensaje}")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

mensajes = [
    "¡GANA 1000€ AHORA! Haz clic aquí, oferta por tiempo limitado!!!",
    "Hola, ¿quedamos el martes para tomar un café?",
    "Hereda millones de un príncipe nigeriano, ¡envía tus datos!",
    "¿Has visto el partido de ayer? Fue increíble el gol en el minuto 90.",
]

for msg in mensajes:
    es_spam = cadena.invoke({"mensaje": msg})
    estado = "🚨 SPAM" if es_spam else "✅ OK  "
    print(f"{estado} → {msg[:55]}...")

# 🚨 SPAM → ¡GANA 1000€ AHORA! Haz clic aquí, oferta por...
# ✅ OK   → Hola, ¿quedamos el martes para tomar un café?...
# 🚨 SPAM → Hereda millones de un príncipe nigeriano, ¡enví...
# ✅ OK   → ¿Has visto el partido de ayer? Fue increíble el...
```

**Personalizar los valores verdadero/falso:**

```python
# Puedes configurar los valores que el parser reconoce como True/False
parser_personalizado = BooleanOutputParser(
    true_val="POSITIVO",
    false_val="NEGATIVO"
)

# En ese caso el prompt debe indicar al modelo responder "POSITIVO" o "NEGATIVO"
```

---

### `DatetimeOutputParser` — Texto → `datetime`
```{index} DatetimeOutputParser
```

Parsea cualquier respuesta del modelo que contenga una fecha/hora y la convierte en un objeto `datetime` de Python.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import DatetimeOutputParser
from langchain_core.prompts import ChatPromptTemplate

parser = DatetimeOutputParser()

# Ver el formato que el parser espera por defecto:
print(parser.get_format_instructions())
# → 'Write a datetime string that matches the following pattern:
#    "%Y-%m-%dT%H:%M:%S.%fZ".
#    Examples: 1132-06-09T00:45:21.019257Z, 1187-12-04T11:36:39.086472Z'

prompt = ChatPromptTemplate.from_messages([
    ("human", "{pregunta}\n{format_instructions}")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

# Ejemplo 1: Fecha histórica
estreno = cadena.invoke({"pregunta": "¿Cuándo se estrenó la película Titanic?"})
print(type(estreno))   # <class 'datetime.datetime'>
print(estreno)         # 1997-12-19 00:00:00
print(estreno.year)    # 1997
print(estreno.strftime("%d de %B de %Y"))  # 19 de December de 1997

# Ejemplo 2: Usar el resultado en cálculos reales
from datetime import datetime

independencia = cadena.invoke({
    "pregunta": "¿Cuándo se produjo el Dos de Mayo de 1808 en Madrid?"
})
años_transcurridos = datetime.now().year - independencia.year
print(f"Han pasado {años_transcurridos} años desde entonces.")
```

**Formato personalizado:**

```python
# Puedes cambiar el formato de entrada esperado
parser_eu = DatetimeOutputParser(format="%d/%m/%Y")

print(parser_eu.get_format_instructions())
# → 'Write a datetime string that matches the following pattern: "%d/%m/%Y".'
# → 'Examples: 30/08/1991, 07/05/2001'
```

---

### `EnumOutputParser` — Texto → valor de un Enum
```{index} EnumOutputParser
```

Parsea la respuesta del modelo y la convierte en un miembro de un `Enum` de Python. Perfecto para clasificaciones con categorías fijas.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import EnumOutputParser
from langchain_core.prompts import ChatPromptTemplate
from enum import Enum

# 1. Definimos el Enum con las categorías posibles
class Prioridad(Enum):
    CRITICA = "critica"
    ALTA    = "alta"
    MEDIA   = "media"
    BAJA    = "baja"

# 2. Creamos el parser con el Enum
parser = EnumOutputParser(enum=Prioridad)

print(parser.get_format_instructions())
# → "Select one of the following options: critica, alta, media, baja"

# 3. Prompt y cadena
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres un sistema de ticketing.
Clasifica la prioridad de incidencias de soporte técnico.
{format_instructions}"""),
    ("human", "Incidencia: {incidencia}")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

# 4. Probamos con distintas incidencias
incidencias = [
    "El servidor de producción está caído, no podemos operar.",
    "Un usuario no recuerda su contraseña.",
    "La fuente del logo en la web tiene un pixel de diferencia.",
    "El sistema de facturación devuelve errores en el 30% de los casos.",
]

for inc in incidencias:
    prioridad = cadena.invoke({"incidencia": inc})
    print(f"[{prioridad.value.upper():8}] {inc[:55]}...")

# [CRITICA ] El servidor de producción está caído, no podemos ...
# [BAJA    ] Un usuario no recuerda su contraseña....
# [BAJA    ] La fuente del logo en la web tiene un pixel de di...
# [ALTA    ] El sistema de facturación devuelve errores en el ...
```

**Uso en lógica de negocio:**

```python
# Al ser un Enum real, podemos usarlo en comparaciones y lógica
if prioridad == Prioridad.CRITICA:
    enviar_alerta_oncall()
elif prioridad == Prioridad.ALTA:
    notificar_equipo_slack()
```

---

## Parte 5: Parsers avanzados y de datos

### `PandasDataFrameOutputParser` — Texto → DataFrame
```{index} PandasDataFrameOutputParser
```

Parsea la respuesta del modelo y la convierte en un `DataFrame` de Pandas. Ideal para análisis de datos o cuando necesitas que el modelo genere datos tabulares.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import PandasDataFrameOutputParser
from langchain_core.prompts import ChatPromptTemplate
import pandas as pd

# Proporcionamos un DataFrame de referencia para que el modelo
# sepa exactamente qué columnas y tipos se esperan
df_referencia = pd.DataFrame({
    "pais":       pd.Series(dtype="str"),
    "capital":    pd.Series(dtype="str"),
    "poblacion":  pd.Series(dtype="int"),
    "continente": pd.Series(dtype="str"),
})

parser = PandasDataFrameOutputParser(dataframe=df_referencia)

print(parser.get_format_instructions()[:200])
# → Instrucciones con el formato JSON esperado para reconstruir el DataFrame

prompt = ChatPromptTemplate.from_messages([
    ("system", "Genera datos tabulares precisos.\n{format_instructions}"),
    ("human", "Dame datos de 5 países de {continente} con su capital y población aproximada.")
])

prompt = prompt.partial(format_instructions=parser.get_format_instructions())
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

df = cadena.invoke({"continente": "América del Sur"})

print(type(df))  # <class 'pandas.core.frame.DataFrame'>
print(df)
#          pais      capital  poblacion    continente
# 0      Brasil     Brasilia  215000000  América del Sur
# 1   Argentina  Buenos Aires  45000000  América del Sur
# 2    Colombia      Bogotá    51000000  América del Sur
# 3        Peru        Lima    33000000  América del Sur
# 4       Chile     Santiago   19000000  América del Sur

# Operar con el DataFrame directamente
print(f"\nTotal de población: {df['poblacion'].sum():,}")
print(f"País más poblado: {df.loc[df['poblacion'].idxmax(), 'pais']}")
```

---

### `RegexParser` — Extracción con expresiones regulares
```{index} RegexParser
```

Extrae grupos de una expresión regular aplicada a la respuesta del modelo. Útil cuando el formato de salida es predecible pero informal.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import RegexParser
from langchain_core.prompts import ChatPromptTemplate

# Definimos el patrón regex y los nombres de los grupos de captura
parser = RegexParser(
    regex=r"Acción:\s*(.+)\nMotivo:\s*(.+)\nConfianza:\s*(\d+)%",
    output_keys=["accion", "motivo", "confianza"],
    default_output_key="accion"  # campo por defecto si el regex falla parcialmente
)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Analiza situaciones y responde SIEMPRE con este formato exacto:
Acción: [acción recomendada]
Motivo: [explicación breve]
Confianza: [número del 0 al 100]%"""),
    ("human", "Situación: {situacion}")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

resultado = cadena.invoke({
    "situacion": "Un cliente lleva 3 meses sin pagar y no responde a emails ni llamadas."
})

print(type(resultado))   # <class 'dict'>
print(resultado)
# {
#   'accion': 'Iniciar proceso de reclamación formal por vía legal',
#   'motivo': 'Agotadas las vías de comunicación amistosas durante 3 meses',
#   'confianza': '85'
# }

# Los valores son strings; conversión manual si se necesita
confianza = int(resultado['confianza'])
print(f"Confianza: {confianza}%")
```

---

### `RegexDictParser` — Múltiples patrones regex
```{index} RegexDictParser
```

Permite definir múltiples patrones regex, uno por cada campo de salida. Más flexible que `RegexParser` cuando cada campo tiene un patrón distinto.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import RegexDictParser
from langchain_core.prompts import ChatPromptTemplate

# Cada campo tiene su propio patrón regex independiente
parser = RegexDictParser(
    output_key_to_format={
        "temperatura": r"Temperatura:\s*([\d.]+)°C",
        "humedad":     r"Humedad:\s*(\d+)%",
        "condicion":   r"Condición:\s*(.+?)(?:\n|$)",
        "viento":      r"Viento:\s*([\d.]+)\s*km/h",
    }
)

prompt = ChatPromptTemplate.from_messages([
    ("system", """Proporciona datos meteorológicos con este formato:
Temperatura: X.X°C
Humedad: XX%
Condición: [descripción]
Viento: XX km/h"""),
    ("human", "Dame el clima típico de {ciudad} en verano.")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser

clima = cadena.invoke({"ciudad": "Sevilla"})

print(clima)
# {
#   'temperatura': '38.5',
#   'humedad': '25',
#   'condicion': 'Soleado y muy caluroso',
#   'viento': '15'
# }
```

---

## Parte 6: Parsers de recuperación de errores

Estos parsers actúan como **red de seguridad**: cuando el modelo produce una salida que no puede parsearse, intentan corregirla automáticamente.

### `OutputFixingParser` — Corrección automática con otro LLM
```{index} OutputFixingParser
```

Cuando el parser base falla, llama al modelo de nuevo con el error y la salida original para que la corrija.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import OutputFixingParser
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Producto(BaseModel):
    nombre: str
    precio: float = Field(description="Precio en euros como número decimal")
    disponible: bool = Field(description="True si hay stock, False si no")
    valoracion: int = Field(description="Valoración del 1 al 5", ge=1, le=5)

# Parser base
parser_base = PydanticOutputParser(pydantic_object=Producto)

# Parser con autocorrección: si falla, le pide al LLM que corrija
parser_robusto = OutputFixingParser.from_llm(
    parser=parser_base,
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)
)

# Simulamos una salida malformada del modelo (tipos incorrectos)
salida_rota = '''
{
  "nombre": "Laptop ProMax",
  "precio": "mil doscientos euros",
  "disponible": "sí claro",
  "valoracion": "cuatro estrellas"
}
'''

# OutputFixingParser detecta el error de validación Pydantic
# y llama al modelo para corregirlo automáticamente
resultado = parser_robusto.parse(salida_rota)

print(type(resultado))          # <class '__main__.Producto'>
print(resultado.nombre)         # Laptop ProMax
print(resultado.precio)         # 1200.0   ← corregido de "mil doscientos euros"
print(resultado.disponible)     # True     ← corregido de "sí claro"
print(resultado.valoracion)     # 4        ← corregido de "cuatro estrellas"
```

**Flujo interno del `OutputFixingParser`:**


```
Modelo genera: {"precio": "mil doscientos euros"}
         ↓
Parser base falla (ValidationError: precio debe ser float)
         ↓
OutputFixingParser llama al LLM:
  "Esta salida: {...} falló con este error: {...}. Corrígela."
         ↓
LLM corrige: {"precio": 1200.0}
         ↓
Parser base → objeto Pydantic ✅
```

---

### `RetryOutputParser` — Reintento con el prompt original
```{index} RetryOutputParser
```

A diferencia de `OutputFixingParser`, este parser reinvoca el modelo con el **prompt original completo** más una indicación del error. Útil cuando la corrección requiere regenerar la respuesta desde cero.

```python
from langchain_openai import ChatOpenAI
from langchain.output_parsers import RetryOutputParser
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class Analisis(BaseModel):
    puntuacion: int = Field(description="Puntuación del 1 al 10", ge=1, le=10)
    categoria: str = Field(description="Exactamente una de: positivo, negativo, neutro")
    resumen: str

parser_base = PydanticOutputParser(pydantic_object=Analisis)

# RetryOutputParser necesita el LLM para reintentar
parser_retry = RetryOutputParser.from_llm(
    parser=parser_base,
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    max_retries=3  # máximo de reintentos antes de lanzar excepción
)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Analiza reseñas.\n{format_instructions}"),
    ("human", "{resena}")
])
prompt = prompt.partial(
    format_instructions=parser_base.get_format_instructions()
)

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Para usar RetryOutputParser necesitamos el prompt_value
prompt_value = prompt.invoke({
    "resena": "El producto llegó roto y el servicio al cliente no respondió."
})

respuesta_raw = modelo.invoke(prompt_value)

# Parseamos con reintento automático en caso de fallo
resultado = parser_retry.parse_with_prompt(
    respuesta_raw.content,
    prompt_value
)

print(resultado.puntuacion)  # 2
print(resultado.categoria)   # negativo
print(resultado.resumen)     # "El cliente tuvo una experiencia muy negativa..."
```

---

### `RetryWithErrorOutputParser` — Reintento con contexto del error
```{index} RetryWithErrorOutputParser
```

Similar a `RetryOutputParser` pero incluye explícitamente el mensaje de error en el prompt de reintento, dando más contexto al modelo para que corrija la respuesta.

```python
from langchain.output_parsers import RetryWithErrorOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel

class Resumen(BaseModel):
    titulo: str
    puntos_clave: list[str]
    conclusion: str

parser_base = PydanticOutputParser(pydantic_object=Resumen)

# Incluye el error en el prompt de reintento para mayor contexto
parser_retry_err = RetryWithErrorOutputParser.from_llm(
    parser=parser_base,
    llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)
)

# Uso igual que RetryOutputParser: parser_retry_err.parse_with_prompt(salida, prompt_value)
```

**Cuándo usar cada parser de recuperación:**

| Parser | Estrategia | Mejor para |
|---|---|---|
| `OutputFixingParser` | Corrige la salida incorrecta | Errores de formato o tipos |
| `RetryOutputParser` | Regenera desde el prompt original | Respuestas completamente incorrectas |
| `RetryWithErrorOutputParser` | Regenera incluyendo el error | Errores complejos que requieren contexto |

---

## Parte 7: Parsers de composición

###  `CombiningOutputParser` — Combina múltiples parsers
```{index} CombiningOutputParser
```

Permite usar varios parsers a la vez y combinar sus resultados en un único diccionario. Cada parser procesa una parte de la respuesta del modelo.

```python
from langchain.output_parsers import CombiningOutputParser
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain.output_parsers import DatetimeOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Combinamos un parser de lista y uno de fecha
parser_lista = CommaSeparatedListOutputParser()
parser_fecha  = DatetimeOutputParser()

parser_combinado = CombiningOutputParser(parsers=[parser_lista, parser_fecha])

print(parser_combinado.get_format_instructions())
# → Instrucciones combinadas: primero la lista separada por comas, luego la fecha

prompt = PromptTemplate(
    template="Responde a ambas preguntas:\n{format_instructions}\n"
             "1. Lista 3 guitarristas de jazz famosos.\n"
             "2. ¿Cuándo nació Miles Davis?",
    input_variables=[],
    partial_variables={"format_instructions": parser_combinado.get_format_instructions()}
)

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo | parser_combinado

resultado = cadena.invoke({})
print(resultado)
# {'output1': ['Wes Montgomery', 'Joe Pass', 'Pat Metheny'],
#  'output2': datetime.datetime(1926, 5, 26, 0, 0)}
```

---

## Parte 8: `with_structured_output` (Método moderno)
```{index} with_structured_output
```

Este es el **enfoque recomendado actualmente**. En lugar de instrucciones en el prompt, aprovecha las capacidades nativas del modelo (function calling, tool use) para garantizar la estructura.

### Ventajas sobre los Output Parsers

- **Más fiable**: el modelo está entrenado para respetar el esquema, no solo instruido.
- **Sin prompt engineering**: no necesitas insertar `format_instructions`.
- **Validación automática**: devuelve directamente un objeto Pydantic validado.
- **Código más limpio y conciso**.

### Uso básico con Pydantic

```python
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Optional

class Pelicula(BaseModel):
    titulo: str = Field(description="Título original de la película")
    año: int = Field(description="Año de estreno")
    director: str = Field(description="Nombre del director")
    generos: List[str] = Field(description="Lista de géneros cinematográficos")
    sinopsis: str = Field(description="Breve sinopsis de no más de 2 frases")
    puntuacion: Optional[float] = Field(default=None, description="Puntuación del 0 al 10")

# Vinculamos el modelo al esquema: sin parsers ni format_instructions
modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
modelo_estructurado = modelo.with_structured_output(Pelicula)

resultado = modelo_estructurado.invoke(
    "Cuéntame sobre la película Inception de Christopher Nolan."
)

print(type(resultado))       # <class '__main__.Pelicula'>
print(resultado.titulo)      # Inception
print(resultado.año)         # 2010
print(resultado.director)    # Christopher Nolan
print(resultado.generos)     # ['Ciencia ficción', 'Thriller', 'Acción']
print(resultado.puntuacion)  # 8.8
```

---

### Con esquemas anidados

```python
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

class Ingrediente(BaseModel):
    nombre: str
    cantidad: str
    unidad: str

class Receta(BaseModel):
    nombre: str = Field(description="Nombre del plato")
    tiempo_preparacion_min: int
    dificultad: str = Field(description="Fácil, Media o Difícil")
    ingredientes: List[Ingrediente]
    pasos: List[str]

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
extractor = modelo.with_structured_output(Receta)

receta = extractor.invoke("Dame una receta de tortilla española para 4 personas.")

print(f"🍳 {receta.nombre} — {receta.dificultad} ({receta.tiempo_preparacion_min} min)")
for ing in receta.ingredientes:
    print(f"  • {ing.cantidad} {ing.unidad} de {ing.nombre}")
```

---

### Con `TypedDict` (sin Pydantic)

```python
from langchain_openai import ChatOpenAI
from typing import TypedDict, List

class ResumenNoticia(TypedDict):
    titular: str
    tema_principal: str
    palabras_clave: List[str]
    sentimiento: str  # positivo, negativo, neutro

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
analizador = modelo.with_structured_output(ResumenNoticia)

resultado = analizador.invoke("""
La economía española creció un 2,5% en el último trimestre, superando las
previsiones de los analistas. El turismo y las exportaciones fueron los
principales motores del crecimiento, según el INE.
""")

print(resultado)
# {'titular': 'Economía española crece 2,5% en el trimestre',
#  'tema_principal': 'Crecimiento económico',
#  'palabras_clave': ['economía', 'crecimiento', 'turismo', 'exportaciones'],
#  'sentimiento': 'positivo'}
```

---

### En cadenas LCEL con `Literal` para valores fijos

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Literal

class AnalisisSentimiento(BaseModel):
    # Literal restringe los valores posibles: el modelo solo puede elegir uno
    sentimiento: Literal["positivo", "negativo", "neutro", "mixto"]
    confianza: float = Field(description="Confianza del 0.0 al 1.0", ge=0.0, le=1.0)
    emociones_detectadas: List[str]
    resumen: str = Field(description="Explicación breve de máximo 20 palabras")

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres experto en análisis de sentimientos en español."),
    ("human", "Analiza: {texto}")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
cadena = prompt | modelo.with_structured_output(AnalisisSentimiento)

textos = [
    "¡Este producto es increíble! Ha superado todas mis expectativas.",
    "El servicio fue pésimo. Una decepción total.",
    "El paquete llegó en el tiempo estimado.",
]

for texto in textos:
    r = cadena.invoke({"texto": texto})
    print(f"[{r.sentimiento.upper():8}] {r.confianza:.0%} → {r.resumen}")

# [POSITIVO] 95% → Experiencia muy satisfactoria y sorprendente
# [NEGATIVO] 92% → Cliente decepcionado con el servicio recibido
# [NEUTRO  ] 88% → Entrega sin incidencias en el plazo previsto
```

---

## Parte 9: Comparativa completa de todos los parsers

### Tabla de referencia rápida

| Parser | Tipo de salida | Paquete | Validación | Caso de uso principal |
|---|---|---|---|---|
| `StrOutputParser` | `str` | `langchain_core` | ❌ | Texto libre, chatbots |
| `CommaSeparatedListOutputParser` | `list[str]` | `langchain_core` | ❌ | Listas simples en una línea |
| `NumberedListOutputParser` | `list[str]` | `langchain_core` | ❌ | Listas numeradas |
| `MarkdownListOutputParser` | `list[str]` | `langchain_core` | ❌ | Listas con `*` o `-` |
| `JsonOutputParser` | `dict` | `langchain_core` | ❌ | JSON sin esquema fijo |
| `SimpleJsonOutputParser` | `dict` | `langchain_core` | ❌ | Alias de JsonOutputParser |
| `PydanticOutputParser` | `BaseModel` | `langchain_core` | ✅ | Esquemas validados con tipos |
| `XMLOutputParser` | `dict` | `langchain_core` | ❌ | Respuestas en formato XML |
| `BooleanOutputParser` | `bool` | `langchain` | ✅ | Clasificación binaria Sí/No |
| `DatetimeOutputParser` | `datetime` | `langchain` | ✅ | Extracción de fechas/horas |
| `EnumOutputParser` | `Enum` | `langchain` | ✅ | Categorías fijas predefinidas |
| `StructuredOutputParser` | `dict` | `langchain` | ❌ | Esquemas simples sin Pydantic |
| `YamlOutputParser` | `BaseModel` | `langchain` | ✅ | Respuestas en formato YAML |
| `PandasDataFrameOutputParser` | `DataFrame` | `langchain` | ✅ | Datos tabulares para análisis |
| `RegexParser` | `dict` | `langchain` | ❌ | Extracción con un patrón regex |
| `RegexDictParser` | `dict` | `langchain` | ❌ | Múltiples patrones regex |
| `OutputFixingParser` | variable | `langchain` | según base | Corrección automática de errores |
| `RetryOutputParser` | variable | `langchain` | según base | Reintento desde el prompt |
| `RetryWithErrorOutputParser` | variable | `langchain` | según base | Reintento con contexto del error |
| `CombiningOutputParser` | `dict` | `langchain` | según base | Combinar múltiples parsers |
| `with_structured_output` | `BaseModel`/`dict` | nativo | ✅ | Enfoque moderno recomendado |

---

### Árbol de decisión para elegir parser

```
¿El modelo soporta function calling/tool use? (GPT-4, Claude, Gemini…)
    │
    ├── SÍ ──→ with_structured_output ✅ (la mejor opción)
    │
    └── NO ──→ Output Parsers
                    │
                    ├── ¿Qué tipo de dato necesitas?
                    │       │
                    │       ├── bool        → BooleanOutputParser
                    │       ├── datetime    → DatetimeOutputParser
                    │       ├── Enum        → EnumOutputParser
                    │       ├── DataFrame   → PandasDataFrameOutputParser
                    │       ├── XML         → XMLOutputParser
                    │       ├── YAML        → YamlOutputParser
                    │       ├── str simple  → StrOutputParser
                    │       │
                    │       └── dict / objeto estructurado
                    │               │
                    │               ├── ¿Necesitas validación de tipos?
                    │               │       ├── SÍ → PydanticOutputParser
                    │               │       └── NO → StructuredOutputParser / JsonOutputParser
                    │               │
                    │               └── ¿Solo lista de items?
                    │                       ├── separados por coma → CommaSeparatedListOutputParser
                    │                       ├── numerada           → NumberedListOutputParser
                    │                       └── markdown (*)       → MarkdownListOutputParser
                    │
                    └── ¿El modelo falla frecuentemente?
                            ├── Corregir salida        → OutputFixingParser
                            ├── Regenerar respuesta    → RetryOutputParser
                            └── Regenerar con error    → RetryWithErrorOutputParser
```

---

## Parte 10: Ejemplo integrador completo

Este ejemplo combina varios parsers y `with_structured_output` en un pipeline realista de análisis de CVs.

```python
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import List, Optional, Literal

# ── Esquema completo del CV ─────────────────────────────────────────────────────

class ExperienciaLaboral(BaseModel):
    empresa: str
    puesto: str
    años: int

class CurriculumVitae(BaseModel):
    nombre: str
    email: Optional[str] = None
    habilidades: List[str]
    experiencia: List[ExperienciaLaboral]
    nivel_ingles: Optional[str] = None
    años_experiencia_total: int = Field(description="Suma total de años trabajados")
    apto_para_entrevista: bool = Field(
        description="True si tiene más de 3 años de experiencia total"
    )
    perfil: Literal["junior", "mid", "senior"] = Field(
        description="junior: <2 años, mid: 2-5 años, senior: >5 años"
    )

# ── Pipeline con with_structured_output ────────────────────────────────────────

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un reclutador experto. Extrae y analiza CVs con precisión."),
    ("human", "Analiza este CV:\n\n{cv_texto}")
])

modelo = ChatOpenAI(model="gpt-4o-mini", temperature=0)
extractor_cv = prompt | modelo.with_structured_output(CurriculumVitae)

# ── CV de prueba ────────────────────────────────────────────────────────────────

cv_texto = """
Carlos Martínez López — carlos.martinez@email.com

EXPERIENCIA
- Desarrollador Senior en TechCorp (4 años): Python, FastAPI, arquitectura cloud
- Junior Developer en StartupXYZ (2 años): React, Node.js

HABILIDADES: Python, FastAPI, React, PostgreSQL, Docker, Git, comunicación efectiva

IDIOMAS: Inglés avanzado (C1)
"""

# ── Ejecución ───────────────────────────────────────────────────────────────────

cv = extractor_cv.invoke({"cv_texto": cv_texto})

print("=" * 52)
print("          INFORME DE CANDIDATO")
print("=" * 52)
print(f"👤 Nombre:       {cv.nombre}")
print(f"📧 Email:        {cv.email}")
print(f"🌍 Inglés:       {cv.nivel_ingles}")
print(f"📊 Perfil:       {cv.perfil.upper()}")
print(f"⏱️  Experiencia:  {cv.años_experiencia_total} años")
print(f"\n💼 Trayectoria:")
for exp in cv.experiencia:
    print(f"   • {exp.puesto} en {exp.empresa} ({exp.años} años)")
print(f"\n🛠️  Habilidades:  {', '.join(cv.habilidades)}")
print(f"\n{'✅ APTO' if cv.apto_para_entrevista else '❌ NO APTO'} para entrevista")
print("=" * 52)
```

**Salida esperada:**

```
====================================================
          INFORME DE CANDIDATO
====================================================
👤 Nombre:       Carlos Martínez López
📧 Email:        carlos.martinez@email.com
🌍 Inglés:       avanzado
📊 Perfil:       SENIOR
⏱️  Experiencia:  6 años

💼 Trayectoria:
   • Desarrollador Senior en TechCorp (4 años)
   • Junior Developer en StartupXYZ (2 años)

🛠️  Habilidades:  Python, FastAPI, React, PostgreSQL, Docker, Git, comunicación efectiva

✅ APTO para entrevista
====================================================
```

---

## Resumen y buenas prácticas

1. **Prefiere `with_structured_output`** cuando el modelo lo soporte (GPT-4, Claude, Gemini). Es más limpio, fiable y requiere menos código.

2. **Usa Pydantic** para definir esquemas. Los `Field(description=...)` son fundamentales: le dan contexto al modelo sobre qué esperar en cada campo.

3. **Usa `Optional` con valores por defecto** para campos que el modelo podría no encontrar, evitando errores de validación inesperados.

4. **Usa `Literal` y `Enum`** para campos con valores fijos. Restringen la salida del modelo de forma muy efectiva y hacen el código más robusto.

5. **Para recuperación de errores**, usa `OutputFixingParser` como primera línea y `RetryOutputParser`/`RetryWithErrorOutputParser` cuando el fallo sea estructural.

6. **`BooleanOutputParser`** es ideal para moderación de contenido, clasificación binaria y sistemas de decisión simple.

7. **`DatetimeOutputParser`** convierte directamente en `datetime` de Python, permitiéndote hacer cálculos de fechas sin conversión manual.

8. **`EnumOutputParser`** garantiza que el modelo solo elige entre tus categorías, lo que hace el código más robusto que comparar
